# Qwen3-14B trên Colab T4 → API cho DACNTT

**MODEL-SERVE-01 · REVIEW · STATIC VALIDATED — NOT GPU TESTED**

Chọn **Runtime → Change runtime type → T4 GPU**, sau đó **Run all** và cấp quyền
mount Google Drive. Notebook chuẩn bị Qwen3-14B **GGUF Q4_K_M (4-bit)**, khởi động
OpenAI-compatible API và thử tạo URL HTTPS. Mặc định context **16384**, một request
inference tại một thời điểm, non-thinking. Chưa xác nhận T4/16K hoạt động thực tế.

Khi thành công, thông tin kết nối tự lưu vào **`MyDrive/DACNTT/api_info.json`**.
API key không hiển thị. Không chia sẻ file này công khai hoặc commit vào Git.

Cloudflare Quick Tunnel chỉ dùng thử trong phiên tương tác, không có uptime bảo đảm,
URL đổi khi start lại và không hỗ trợ SSE: client phải dùng **`stream=false`**.
Chỉ sử dụng khi chính sách runtime/tài khoản cho phép; Colab Free có thể chặn hoặc
kết thúc phiên. Không có cơ chế vượt chặn hoặc giữ phiên sống. Nếu tunnel thất bại,
notebook giữ API local và ghi `BASE_URL: null`, không xuất URL public giả.

[Model chính thức](https://huggingface.co/Qwen/Qwen3-14B-GGUF) ·
[llama-server đã pin](https://github.com/ggml-org/llama.cpp/blob/cd959bae99ff4bed22bef2604f45241c09bbb8c8/tools/server/README.md) ·
[Cloudflare Quick Tunnel](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) ·
[Giới hạn Colab](https://research.google.com/colaboratory/faq.html)

## 1. Chuẩn bị GPU và Google Drive

Giữ cấu hình mặc định để thử T4/16K. Chỉ đổi xuống 8192 nếu gặp CUDA OOM, sau đó
chạy lại từ cell này. Có thể tắt HTTPS để chỉ test local.

In [ ]:
import hashlib
import json
import os
import re
import secrets
import shutil
import signal
import socket
import struct
import subprocess
import time
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
import requests
from google.colab import drive
from IPython.display import display
from tqdm.auto import tqdm

# fmt: off
#@title Chuẩn bị
CONTEXT_SIZE = 16384 #@param [16384, 8192] {type:"raw"}
ENABLE_HTTPS = True #@param {type:"boolean"}
# fmt: on

ROOT = Path("/content/dacntt_qwen")
ROOT.mkdir(exist_ok=True)
MODEL_NAME = "qwen3-14b-t4"
MODEL_REPO = "Qwen/Qwen3-14B-GGUF"
MODEL_REVISION = "530227a7d994db8eca5ab5ced2fb692b614357fd"
MODEL_FILE = "Qwen3-14B-Q4_K_M.gguf"
MODEL_SIZE = 9001752960
PORT = 8000
API = f"http://127.0.0.1:{PORT}"
SERVER_READY = False
BASE_URL = None


def process_identity(pid):
    try:
        return Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[19]
    except FileNotFoundError:
        return None


def stop_process(kind):
    state_file = ROOT / f"{kind}.json"
    if not state_file.exists():
        return
    state = json.loads(state_file.read_text())
    pid = state["pid"]
    if state["starttime"] and process_identity(pid) == state["starttime"]:
        args = Path(f"/proc/{pid}/cmdline").read_bytes().split(b"\0")
        if str(state["binary"]).encode() not in args or os.getpgid(pid) != pid:
            raise RuntimeError("Không xác minh được process sở hữu. Hãy Restart Session.")
        os.killpg(pid, signal.SIGTERM)
        proc = globals().get(kind.upper() + "_PROCESS")
        for _ in range(30):
            if proc is not None and proc.pid == pid:
                proc.poll()
            if process_identity(pid) is None:
                break
            time.sleep(0.1)
        else:
            if process_identity(pid) == state["starttime"]:
                os.killpg(pid, signal.SIGKILL)
        if proc is not None and proc.pid == pid:
            proc.wait(timeout=10)
    state_file.unlink(missing_ok=True)


def save_api_info(status, base_url=None):
    info = {
        "BASE_URL": base_url,
        "API_KEY": API_KEY,
        "MODEL_NAME": MODEL_NAME,
        "STATUS": status,
        "LOCAL_BASE_URL": API + "/v1",
        "UPDATED_AT": datetime.now(UTC).isoformat(),
    }
    temporary = INFO_PATH.with_suffix(".tmp")
    temporary.write_text(json.dumps(info, indent=2), encoding="utf-8")
    os.replace(temporary, INFO_PATH)


def stop_api():
    global SERVER_READY, BASE_URL
    SERVER_READY, BASE_URL = False, None
    stop_process("tunnel")
    stop_process("server")
    if "INFO_PATH" in globals() and "API_KEY" in globals():
        save_api_info("STOPPED")


stop_api()
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/DACNTT")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
INFO_PATH = DRIVE_ROOT / "api_info.json"
API_KEY = secrets.token_urlsafe(32)
HEADERS = {"Authorization": "Bearer " + API_KEY}
# Xóa URL cũ khỏi bản ghi trước khi chuẩn bị phiên mới; không in API key.
save_api_info("PREPARING")


def gpu_memory():
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used,memory.free,compute_cap",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
        timeout=15,
    )
    fields = [v.strip() for v in result.stdout.splitlines()[0].split(",")]
    return dict(
        zip(["name", "total_MiB", "used_MiB", "free_MiB", "compute_cap"], fields, strict=True)
    )


try:
    GPU = gpu_memory()
except (OSError, subprocess.SubprocessError):
    raise RuntimeError("Không có NVIDIA GPU. Chọn T4 GPU rồi chạy lại.") from None
CUDA_ARCH = GPU["compute_cap"].replace(".", "")
print("GPU:", GPU)
if "T4" not in GPU["name"]:
    print("Cảnh báo: runtime được cấp GPU khác T4; kết quả áp dụng cho GPU trên.")
print("Context mục tiêu:", CONTEXT_SIZE, "| GGUF Q4_K_M | non-thinking")

## 2. Chuẩn bị model và CUDA server

Ưu tiên cache **`MyDrive/DACNTT/models/Qwen3-14B-Q4_K_M/`**, copy về `/content/models/`
để load nhanh. Chỉ kiểm tra kích thước và GGUF header, **không hash toàn bộ model**;
đây không phải xác minh toàn vẹn cryptographic. Nếu cache đúng kích thước nhưng load
báo hỏng, đổi tên file cache rồi chạy lại cell để tải mới.

Model khoảng 8,4 GiB. Khi Drive thiếu quota vẫn có thể dùng bản local.
Build llama.cpp CUDA đã pin có thể mất vài phút; binary hợp lệ trong phiên được dùng lại.

Nếu đã có compiler/CMake/Ninja/Git (thường có trên Colab), cell build bỏ qua apt.
Nếu thiếu, chỉ cài dependency còn thiếu từ nguồn Ubuntu chính thức với timeout,
không truy vấn các PPA không liên quan hoặc sửa cấu hình apt của runtime.
Sau lỗi apt cũ: mở bản notebook mới và chạy lại riêng cell **Build llama-server CUDA**
trong cùng phiên; không cần tải lại model hay reset session.


In [ ]:
# @title Chuẩn bị model
stop_api()
save_api_info("PREPARING")
LOCAL_DIR = Path("/content/models")
LOCAL_DIR.mkdir(exist_ok=True)
MODEL_PATH = LOCAL_DIR / MODEL_FILE
DRIVE_DIR = DRIVE_ROOT / "models/Qwen3-14B-Q4_K_M"
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CACHED_MODEL = DRIVE_DIR / MODEL_FILE


def model_ok(path):
    if not path.is_file() or path.stat().st_size != MODEL_SIZE:
        return False
    with path.open("rb") as stream:
        magic, version, tensors, metadata = struct.unpack("<4sIQQ", stream.read(24))
    return magic == b"GGUF" and version == 3 and tensors > 0 and metadata > 0


def require_space(directory):
    if shutil.disk_usage(directory).free < MODEL_SIZE + 512 * 2**20:
        raise OSError(f"Không đủ disk space tại {directory}.")


def copy_model(source, destination):
    require_space(destination.parent)
    partial = destination.with_suffix(".copying")
    try:
        print("Copy model →", destination)
        shutil.copyfile(source, partial)
        if not model_ok(partial):
            raise ValueError("Copy không khớp kích thước/GGUF header.")
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)


if model_ok(MODEL_PATH):
    print("Dùng model đã có ở local.")
else:
    copied = False
    try:
        if model_ok(CACHED_MODEL):
            print("Tìm thấy cache Google Drive.")
            copy_model(CACHED_MODEL, MODEL_PATH)
            copied = True
    except OSError as exc:
        print("Không đọc/copy được Drive:", type(exc).__name__, "→ tải vào local.")
    if not copied:
        require_space(LOCAL_DIR)
        url = f"https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{MODEL_FILE}"
        partial = MODEL_PATH.with_suffix(".downloading")
        deadline = time.monotonic() + 3600
        try:
            with requests.get(url, stream=True, timeout=(20, 120)) as response:
                if response.status_code != 200:
                    raise RuntimeError(
                        f"Hugging Face HTTP {response.status_code}; chưa tải được model."
                    )
                with (
                    partial.open("wb") as stream,
                    tqdm(total=MODEL_SIZE, unit="B", unit_scale=True) as bar,
                ):
                    for chunk in response.iter_content(8 * 2**20):
                        if time.monotonic() > deadline:
                            raise TimeoutError("Download quá 1 giờ. Chạy lại thủ công.")
                        stream.write(chunk)
                        bar.update(len(chunk))
                        if stream.tell() > MODEL_SIZE:
                            raise ValueError("Download vượt kích thước model.")
            if not model_ok(partial):
                raise ValueError("Model tải chưa đủ hoặc GGUF header không hợp lệ.")
            os.replace(partial, MODEL_PATH)
        finally:
            partial.unlink(missing_ok=True)
try:
    if not model_ok(CACHED_MODEL):
        copy_model(MODEL_PATH, CACHED_MODEL)
except OSError as exc:
    print("Không lưu được weights lên Drive:", type(exc).__name__, "→ dùng local trong phiên này.")
print("Model sẵn sàng:", MODEL_PATH)

In [ ]:
import platform

# @title Build llama-server CUDA (dùng lại binary nếu hợp lệ)
LLAMA_COMMIT = "cd959bae99ff4bed22bef2604f45241c09bbb8c8"
SOURCE = ROOT / "llama.cpp"
BINARY = SOURCE / "build/bin/llama-server"
BUILD_STAMP = ROOT / "build.json"
BUILD_LOG = ROOT / "build.log"
expected_build = {"commit": LLAMA_COMMIT, "cuda_arch": CUDA_ARCH}


def run_build(command, timeout=1200):
    print("Lệnh:", " ".join(map(str, command)))
    with BUILD_LOG.open("a") as log:
        try:
            subprocess.run(
                list(map(str, command)),
                stdout=log,
                stderr=subprocess.STDOUT,
                timeout=timeout,
                check=True,
            )
        except (subprocess.SubprocessError, OSError) as exc:
            print(BUILD_LOG.read_text(errors="replace")[-6000:])
            reason = (
                f"timeout sau {timeout} giây"
                if isinstance(exc, subprocess.TimeoutExpired)
                else str(exc)
            )
            raise RuntimeError(f"Lệnh thất bại: {reason}; xem lệnh trên và {BUILD_LOG}") from None


def ensure_build_tools():
    tools = {
        "gcc": "build-essential",
        "g++": "build-essential",
        "make": "build-essential",
        "cmake": "cmake",
        "ninja": "ninja-build",
        "git": "git",
    }
    missing = sorted({package for tool, package in tools.items() if not shutil.which(tool)})
    if not missing:
        print("Đã có đủ build tools; bỏ qua apt-get update/install.")
        return
    print("Cần cài build dependencies:", ", ".join(missing))
    release = platform.freedesktop_os_release()
    codename = release.get("VERSION_CODENAME", "")
    if release.get("ID") != "ubuntu" or not re.fullmatch(r"[a-z]+", codename):
        raise RuntimeError(
            "Tự cài build tools chỉ hỗ trợ Ubuntu Colab; không thay apt sources hệ thống."
        )
    sources = ROOT / "ubuntu-build.list"
    empty_parts = ROOT / "apt-build-sourceparts"
    empty_parts.mkdir(exist_ok=True)
    sources.write_text(
        "".join(
            f"deb [signed-by=/usr/share/keyrings/ubuntu-archive-keyring.gpg] "
            f"{host} {suite} main universe\n"
            for host, suite in [
                ("https://archive.ubuntu.com/ubuntu", codename),
                ("https://archive.ubuntu.com/ubuntu", codename + "-updates"),
                ("https://security.ubuntu.com/ubuntu", codename + "-security"),
            ]
        )
    )
    # Chỉ áp dụng cho hai lệnh này, không sửa/xóa PPA, CUDA hay cấu hình apt của Colab.
    apt = [
        "apt-get",
        "-o",
        f"Dir::Etc::sourcelist={sources}",
        "-o",
        f"Dir::Etc::sourceparts={empty_parts}",
        "-o",
        "Acquire::Retries=0",
        "-o",
        "Acquire::https::Timeout=20",
        "-o",
        "Acquire::http::Timeout=20",
    ]
    run_build([*apt, "update"], 180)
    run_build([*apt, "install", "-y", "--no-install-recommends", *missing], 600)
    if any(not shutil.which(tool) for tool in tools):
        raise RuntimeError("Cài dependency xong nhưng vẫn thiếu build tools; xem build.log.")


def cuda_binary_ok():
    if not BINARY.is_file():
        return False
    result = subprocess.run(
        [str(BINARY), "--list-devices"], capture_output=True, text=True, timeout=30
    )
    return result.returncode == 0 and bool(re.search(r"CUDA\d", result.stdout + result.stderr))


if not (
    BUILD_STAMP.exists()
    and json.loads(BUILD_STAMP.read_text()) == expected_build
    and cuda_binary_ok()
):
    stop_api()
    if not shutil.which("nvcc"):
        raise RuntimeError("CUDA toolkit nvcc MISSING. Chọn Colab GPU runtime và Restart Session.")
    BUILD_LOG.write_text("")
    ensure_build_tools()
    if not SOURCE.exists():
        run_build(["git", "init", SOURCE], 60)
        run_build(
            [
                "git",
                "-C",
                SOURCE,
                "remote",
                "add",
                "origin",
                "https://github.com/ggml-org/llama.cpp.git",
            ],
            60,
        )
    run_build(["git", "-C", SOURCE, "fetch", "--depth", "1", "origin", LLAMA_COMMIT], 300)
    run_build(["git", "-C", SOURCE, "checkout", "--detach", LLAMA_COMMIT], 60)
    run_build(
        [
            "cmake",
            "-S",
            SOURCE,
            "-B",
            SOURCE / "build",
            "-G",
            "Ninja",
            "-DGGML_CUDA=ON",
            f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",
            "-DCMAKE_BUILD_TYPE=Release",
            "-DLLAMA_CURL=OFF",
            "-DLLAMA_OPENSSL=OFF",
            "-DLLAMA_BUILD_TESTS=OFF",
            "-DLLAMA_BUILD_EXAMPLES=OFF",
            "-DLLAMA_BUILD_SERVER=ON",
        ]
    )
    run_build(["cmake", "--build", SOURCE / "build", "--target", "llama-server", "-j", "2"])
    if not cuda_binary_ok():
        raise RuntimeError("FAILED: binary không liệt kê CUDA device; không chạy CPU fallback.")
    BUILD_STAMP.write_text(json.dumps(expected_build))
print("CUDA backend/device: verified qua --list-devices; chưa chứng minh GPU inference.")
print(subprocess.run([str(BINARY), "--version"], capture_output=True, text=True, timeout=15).stdout)

## 3. Start API

Cell này dừng server/tunnel cũ của notebook, khởi động server mới và kiểm tra inference
local. Nếu bật HTTPS, thử Cloudflare Tunnel trong tối đa 90 giây rồi kiểm tra `/v1/models`
qua URL public và xác nhận request thiếu API key bị từ chối.

Chỉ sau khi kiểm tra thành công, `api_info.json` có `STATUS=READY` và `BASE_URL` HTTPS
kết thúc bằng `/v1`. Kiểm tra public được thực hiện **từ Colab**, chưa xác minh từ máy
chạy DACNTT. Codex có thể đọc file khi được kết nối/quyền truy cập Drive tương ứng
hoặc qua Drive sync; mount Colab không tự chia sẻ filesystem với máy Codex.

Nếu CUDA OOM: chọn 8192 ở bước 1 rồi chạy lại. Không tự retry load model.

In [ ]:
# @title Start API
stop_api()
save_api_info("STARTING")
if not model_ok(MODEL_PATH) or not cuda_binary_ok():
    raise RuntimeError("Chạy bước 2 thành công trước.")
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError("Port 8000 đang được process khác dùng. Restart Session rồi chạy lại.")


def start_process(kind, command, env=None):
    with (ROOT / f"{kind}.log").open("w") as log:
        proc = subprocess.Popen(
            command, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True
        )
    (ROOT / f"{kind}.json").write_text(
        json.dumps({"pid": proc.pid, "starttime": process_identity(proc.pid), "binary": command[0]})
    )
    globals()[kind.upper() + "_PROCESS"] = proc
    return proc


def api_json(route, payload=None, base_url=None, timeout=180):
    url = (base_url or API + "/v1") + route
    response = requests.request(
        "GET" if payload is None else "POST",
        url,
        headers=HEADERS,
        json=payload,
        timeout=(10, timeout),
    )
    if not response.ok:
        raise RuntimeError(f"HTTP {response.status_code} tại API; xem server.log/tunnel.log.")
    return response.json()


def chat_payload(prompt, **extra):
    return {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": 256,
        "stream": False,
        "reasoning_effort": "none",
        "chat_template_kwargs": {"enable_thinking": False},
        **extra,
    }


def final_content(data):
    message = data["choices"][0]["message"]
    content = message.get("content")
    if (
        not content
        or message.get("reasoning_content")
        or message.get("reasoning")
        or "<think>" in content
        or "</think>" in content
    ):
        raise ValueError("Thiếu final content hoặc model chưa tắt thinking.")
    if data["choices"][0].get("finish_reason") == "length":
        raise ValueError("Output bị cắt vì hết token budget.")
    return content


child_env = {
    k: v for k, v in os.environ.items() if not k.startswith("LLAMA_ARG_") and k != "LLAMA_API_KEY"
}
child_env["LLAMA_API_KEY"] = API_KEY
command = [
    str(BINARY),
    "--model",
    str(MODEL_PATH),
    "--alias",
    MODEL_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--ctx-size",
    str(CONTEXT_SIZE),
    "--parallel",
    "1",
    "--n-gpu-layers",
    "999",
    "--jinja",
    "--reasoning",
    "off",
    "--reasoning-format",
    "deepseek",
    "--batch-size",
    "512",
    "--ubatch-size",
    "128",
]
try:
    SERVER_PROCESS = start_process("server", command, child_env)
    deadline = time.monotonic() + 600
    for _ in range(300):
        if SERVER_PROCESS.poll() is not None:
            raise RuntimeError("llama-server đã thoát; xem log bên dưới.")
        try:
            if requests.get(API + "/health", timeout=2).status_code == 200:
                break
        except requests.RequestException:
            pass
        if time.monotonic() >= deadline:
            raise TimeoutError("Model chưa sẵn sàng sau 600 giây.")
        time.sleep(2)
    else:
        raise TimeoutError("Model readiness timeout.")
    log_text = (ROOT / "server.log").read_text(errors="replace")
    offload = re.findall(r"offloaded\s+(\d+)/(\d+)\s+layers", log_text)
    if not offload or int(offload[-1][0]) == 0:
        raise RuntimeError("Chưa xác nhận GPU offload trong server.log.")
    models = api_json("/models")
    if MODEL_NAME not in [m["id"] for m in models.get("data", [])]:
        raise ValueError("API không trả model alias đã cấu hình.")
    props = api_json("/props", base_url=API)
    if props.get("default_generation_settings", {}).get("n_ctx") != CONTEXT_SIZE:
        raise ValueError("Context thực tế khác cấu hình yêu cầu.")
    print(
        "Model trả lời:",
        final_content(
            api_json("/chat/completions", chat_payload("Chào bạn! Trả lời ngắn bằng tiếng Việt."))
        ),
    )
    SERVER_READY = True
    print("LOCAL READY:", API + "/v1", "| Context:", CONTEXT_SIZE, "| GPU:", GPU["name"])
except BaseException:
    stop_api()
    save_api_info("FAILED")
    tail = (ROOT / "server.log").read_text(errors="replace")[-5000:]
    print(tail.replace(API_KEY, "<REDACTED>"))
    print("Nếu CUDA OOM: chọn 8192 ở bước 1 rồi chạy lại. Không tự retry.")
    raise

BASE_URL = None
if ENABLE_HTTPS:
    try:
        cloudflared = ROOT / "cloudflared"
        expected_hash = "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872"
        # Chỉ hash binary tunnel ~40 MB; không hash weights ~9 GB.
        if (
            not cloudflared.exists()
            or hashlib.sha256(cloudflared.read_bytes()).hexdigest() != expected_hash
        ):
            url = "https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64"
            response = requests.get(url, timeout=(15, 120))
            if (
                response.status_code != 200
                or hashlib.sha256(response.content).hexdigest() != expected_hash
            ):
                raise RuntimeError("Không tải/xác minh được cloudflared đã pin.")
            cloudflared.write_bytes(response.content)
            cloudflared.chmod(0o755)
        TUNNEL_PROCESS = start_process(
            "tunnel",
            [str(cloudflared), "tunnel", "--url", API, "--no-autoupdate", "--protocol", "http2"],
        )
        deadline = time.monotonic() + 90
        for _ in range(45):
            if TUNNEL_PROCESS.poll() is not None:
                raise RuntimeError("Cloudflare Tunnel đã thoát.")
            log_text = (ROOT / "tunnel.log").read_text(errors="replace")
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
            if match:
                candidate = match[0] + "/v1"
                try:
                    models = api_json("/models", base_url=candidate, timeout=5)
                    denied = requests.get(candidate + "/models", timeout=(5, 5))
                    if MODEL_NAME in [
                        m["id"] for m in models.get("data", [])
                    ] and denied.status_code in (401, 403):
                        BASE_URL = candidate
                        break
                except (requests.RequestException, ValueError, RuntimeError):
                    pass
            if time.monotonic() >= deadline:
                break
            time.sleep(2)
        if BASE_URL is None:
            raise TimeoutError("Chưa xác minh được HTTPS API trong 90 giây.")
    except Exception as exc:
        stop_process("tunnel")
        print(
            "HTTPS chưa khả dụng:",
            str(exc).replace(API_KEY, "<REDACTED>")[:500],
            "— chỉ dùng local; xem tunnel.log.",
        )
try:
    save_api_info("READY" if BASE_URL else "LOCAL_ONLY", BASE_URL)
except OSError:
    stop_api()
    raise RuntimeError("Không lưu được api_info.json trên Drive; đã dừng API.") from None
print("BASE_URL:", BASE_URL or "UNAVAILABLE — local API only")
print("Đã lưu BASE_URL, API_KEY (ẩn), MODEL_NAME tại:", INFO_PATH)
print("URL chỉ có hiệu lực khi runtime/server/tunnel còn chạy.")

## 4. Test API

Hai request tuần tự: hỏi tiếng Việt và JSON với số liệu mẫu. Dùng HTTPS nếu đã xác minh,
nếu chưa có thì test local và ghi rõ. Tốc độ lấy từ backend; `UNKNOWN` khi không có số đo.
VRAM là snapshot sau request, không phải peak. Đây là smoke API, không phải benchmark
hay bằng chứng chất lượng dữ liệu tài chính. Không test dài để tiết kiệm Colab Free.

In [ ]:
# @title Test API
if not SERVER_READY:
    raise RuntimeError("Start API thành công trước khi test.")
TEST_BASE = BASE_URL or API + "/v1"
print("Kiểm tra qua:", TEST_BASE)


def check_answer(content):
    if "hà nội" not in content.casefold():
        raise ValueError("Chưa nhận được đáp án Hà Nội.")


def check_json(content):
    value = json.loads(content)
    expected = {"metric": "gross_margin", "numerator": 100, "denominator": 1000, "percentage": 10}
    if not isinstance(value, dict) or not expected.keys() <= value.keys():
        raise ValueError("Thiếu JSON keys.")
    if value["metric"] != "gross_margin" or any(
        type(value[k]) not in (int, float) or value[k] != expected[k]
        for k in ("numerator", "denominator", "percentage")
    ):
        raise ValueError("JSON sai kiểu số hoặc giá trị.")


tests = [
    ("Tiếng Việt", chat_payload("Thủ đô Việt Nam là thành phố nào? Trả lời ngắn."), check_answer),
    (
        "JSON",
        chat_payload(
            'Chỉ trả JSON: metric="gross_margin", numerator=100, denominator=1000, percentage=10.',
            response_format={"type": "json_object"},
        ),
        check_json,
    ),
]
rows = []
for name, payload, checker in tests:
    row = {
        "Test": name,
        "Status": "FAIL",
        "Input tokens": "UNKNOWN",
        "Output tokens": "UNKNOWN",
        "Generation tok/s": "UNKNOWN",
        "GPU used MiB": "UNKNOWN",
        "GPU free MiB": "UNKNOWN",
        "Error": "",
    }
    started = time.perf_counter()
    try:
        data = api_json("/chat/completions", payload, base_url=TEST_BASE, timeout=180)
        usage = data.get("usage") or {}
        row["Input tokens"] = usage.get("prompt_tokens", "UNKNOWN")
        row["Output tokens"] = usage.get("completion_tokens", "UNKNOWN")
        row["Generation tok/s"] = (data.get("timings") or {}).get("predicted_per_second", "UNKNOWN")
        content = final_content(data)
        print(name, "→", content[:1000])
        checker(content)
        row["Status"] = "PASS"
    except requests.Timeout:
        stop_api()
        row["Error"] = "Request timeout: đã dừng API để tránh inference tiếp tục ngầm."
    except Exception as exc:
        row["Error"] = str(exc).replace(API_KEY, "<REDACTED>")[:500]
    row["Latency s"] = round(time.perf_counter() - started, 3)
    memory = gpu_memory()
    row["GPU used MiB"], row["GPU free MiB"] = memory["used_MiB"], memory["free_MiB"]
    rows.append(row)
    if not SERVER_READY:
        break
display(pd.DataFrame(rows).fillna("UNKNOWN"))

### Kết thúc phiên

**Run all không dừng API**. Khi dùng xong, bật ô Stop rồi chạy riêng cell dưới.
Restart Session: Runtime → Disconnect and delete runtime; sau đó chạy từ đầu.
Khi phiên bị ngắt đột ngột, `api_info.json` có thể còn URL cũ: client cần kiểm tra
`/v1/models` trước khi dùng; chạy lại Start API sẽ tự ghi thông tin mới.
Nếu lỗi, gửi traceback/đoạn cuối log đã che secrets, **không gửi `api_info.json`**.

In [ ]:
# fmt: off
#@title Stop API (chỉ chạy khi kết thúc)
STOP_API = False #@param {type:"boolean"}
# fmt: on
if STOP_API:
    stop_api()
    print("Đã dừng server/tunnel và đánh dấu STOPPED trong api_info.json.")